# AutoSuggest Fine-Tuning on Google Colab

Fine-tune a language model on your AutoSuggest typing data using **Unsloth** for 2-5x faster training.

**Requirements:**
- Google Colab with GPU (free T4 works, A100 is faster)
- Your exported training data (`training-pairs.jsonl`) from AutoSuggest

**What this notebook does:**
1. Install dependencies
2. Upload your training data
3. Prepare data in the right format
4. Fine-tune with LoRA (fast, memory-efficient)
5. Export to GGUF + Ollama Modelfile
6. Download your fine-tuned model

**Time estimate:** ~10-30 minutes depending on data size and GPU.

## Step 1: Setup & Install

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
%%capture
# Install Unsloth (optimized for Colab)
!pip install unsloth[colab-new]
!pip install --no-deps trl peft accelerate bitsandbytes xformers
print("Installation complete!")

## Step 2: Upload Your Training Data

Export your training data from AutoSuggest:
1. Open AutoSuggest > Settings > Privacy
2. Click "Export Training Data"
3. Upload the `.jsonl` file below

In [ ]:
from google.colab import files
import json
from pathlib import Path

print("Upload your AutoSuggest training data (.jsonl file):")
uploaded = files.upload()

# Find the uploaded file
input_file = list(uploaded.keys())[0]
print(f"\nUploaded: {input_file}")

# Count and validate pairs
pairs = []
with open(input_file) as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                obj = json.loads(line)
                if obj.get("prompt") and obj.get("completion"):
                    pairs.append(obj)
            except json.JSONDecodeError:
                pass

print(f"Valid training pairs: {len(pairs)}")
if len(pairs) < 10:
    print("Warning: Very small dataset. Results may not be great.")
elif len(pairs) < 100:
    print("Note: Small dataset. Fine-tuning works best with 500+ pairs.")
else:
    print("Good dataset size for fine-tuning!")

# Show a sample
if pairs:
    print(f"\nSample pair:")
    print(f"  Prompt:     {pairs[0]['prompt'][:80]}...")
    print(f"  Completion: {pairs[0]['completion'][:80]}")

## Step 3: Configuration

Choose your model and training settings. **Defaults work well for most users.**

In [ ]:
#@title Training Configuration { display-mode: "form" }

#@markdown ### Model Selection
model_preset = "qwen2.5-1.5b" #@param ["qwen2.5-0.5b", "qwen2.5-1.5b", "qwen2.5-3b", "qwen2.5-7b", "llama3.2-1b", "llama3.2-3b"]

#@markdown ### Training Settings
num_epochs = 3 #@param {type: "integer"}
learning_rate = 2e-4 #@param {type: "number"}
batch_size = 4 #@param {type: "integer"}

#@markdown ### LoRA Settings
lora_rank = 16 #@param {type: "integer"}
lora_alpha = 32 #@param {type: "integer"}

#@markdown ### Export Settings
quantization = "q4_k_m" #@param ["q4_k_m", "q5_k_m", "q8_0", "f16"]
ollama_model_name = "autosuggest-finetuned" #@param {type: "string"}

# Model ID mapping
MODEL_MAP = {
    "qwen2.5-0.5b": "unsloth/Qwen2.5-0.5B-Instruct",
    "qwen2.5-1.5b": "unsloth/Qwen2.5-1.5B-Instruct",
    "qwen2.5-3b": "unsloth/Qwen2.5-3B-Instruct",
    "qwen2.5-7b": "unsloth/Qwen2.5-7B-Instruct",
    "llama3.2-1b": "unsloth/Llama-3.2-1B-Instruct",
    "llama3.2-3b": "unsloth/Llama-3.2-3B-Instruct",
}

model_id = MODEL_MAP[model_preset]
max_seq_length = 4096 if "7b" in model_preset else 2048

print(f"Model:          {model_id}")
print(f"Epochs:         {num_epochs}")
print(f"Learning rate:  {learning_rate}")
print(f"Batch size:     {batch_size}")
print(f"LoRA rank:      {lora_rank}")
print(f"Quantization:   {quantization}")
print(f"Ollama name:    {ollama_model_name}")

## Step 4: Prepare Training Data

In [ ]:
import random

SYSTEM_PROMPT = (
    "You are an autocomplete engine. Complete the user's text naturally. "
    "Only output the completion, nothing else."
)

# Format as chat messages
formatted = []
for pair in pairs:
    formatted.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": pair["prompt"]},
            {"role": "assistant", "content": pair["completion"]},
        ]
    })

# Shuffle and split
random.seed(42)
random.shuffle(formatted)
split_idx = max(1, int(len(formatted) * 0.9))
train_data = formatted[:split_idx]
val_data = formatted[split_idx:]

# Write files
Path("data").mkdir(exist_ok=True)
for name, data in [("train", train_data), ("val", val_data)]:
    with open(f"data/{name}.jsonl", "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

print(f"Training samples:   {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

## Step 5: Fine-Tune the Model

This is the main training step. Takes ~10-30 minutes on a free T4 GPU.

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# Load model
print(f"Loading {model_id}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_alpha,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model loaded and LoRA applied!")
model.print_trainable_parameters()

In [ ]:
# Load prepared dataset
dataset = load_dataset("json", data_files={
    "train": "data/train.jsonl",
    "validation": "data/val.jsonl",
})

# Apply chat template
def apply_chat_template(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(apply_chat_template, batched=True)
print(f"Dataset ready: {len(dataset['train'])} train, {len(dataset['validation'])} val")
print(f"\nSample formatted text:\n{dataset['train'][0]['text'][:500]}")

In [ ]:
# Train!
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    args=TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        fp16=True,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="output/checkpoints",
        save_strategy="epoch",
        eval_strategy="epoch",
        report_to="none",
    ),
)

print("Starting training...\n")
stats = trainer.train()
print(f"\n{'='*50}")
print(f"Training complete!")
print(f"Final loss: {stats.training_loss:.4f}")
print(f"{'='*50}")

In [ ]:
# Save adapter
model.save_pretrained("output/adapter")
tokenizer.save_pretrained("output/adapter")
print("LoRA adapter saved!")

## Step 6: Test the Fine-Tuned Model

In [ ]:
# Quick test
FastLanguageModel.for_inference(model)

test_prompts = [
    "Thanks for the",
    "I'll follow up with",
    "def calculate_",
    "The meeting is scheduled for",
]

print("Testing fine-tuned model:\n")
for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=64, temperature=0.3, top_p=0.9
    )
    completion = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"  \"{prompt}\" → \"{completion.strip()}\"")

## Step 7: Export to GGUF + Ollama

In [ ]:
# Export to GGUF
print(f"Exporting GGUF with {quantization} quantization...")
print("This may take a few minutes.\n")

model.save_pretrained_gguf(
    "output/gguf",
    tokenizer,
    quantization_method=quantization,
)

import glob
gguf_files = glob.glob("output/gguf/*.gguf")
if gguf_files:
    gguf_file = gguf_files[0]
    import os
    size_mb = os.path.getsize(gguf_file) / (1024 * 1024)
    print(f"GGUF model: {gguf_file} ({size_mb:.0f} MB)")
else:
    print("Warning: No GGUF file found. Check output above for errors.")

In [ ]:
# Create Ollama Modelfile
import os

gguf_files = glob.glob("output/gguf/*.gguf")
gguf_filename = os.path.basename(gguf_files[0]) if gguf_files else "model.gguf"

modelfile_content = f"""# AutoSuggest Fine-Tuned Model
# Usage: ollama create {ollama_model_name} -f Modelfile

FROM {gguf_filename}

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER stop "<|endoftext|>"
PARAMETER stop "<|im_end|>"
PARAMETER num_predict 128

SYSTEM \"\"\"You are an autocomplete engine. Complete the user's text naturally. Only output the completion, nothing else.\"\"\"
"""

with open("output/gguf/Modelfile", "w") as f:
    f.write(modelfile_content)

print("Ollama Modelfile created!")
print(f"\nAfter downloading, run:")
print(f"  cd <download-folder>")
print(f"  ollama create {ollama_model_name} -f Modelfile")
print(f"  # Then set '{ollama_model_name}' as your model in AutoSuggest")

## Step 8: Download Your Model

Download the GGUF model + Modelfile, then import into Ollama on your Mac.

In [ ]:
# Create a zip of the GGUF output for easy download
import shutil

shutil.make_archive("autosuggest-model", "zip", "output/gguf")
print("Created autosuggest-model.zip")
print(f"Contents: GGUF model + Ollama Modelfile\n")

# Download
files.download("autosuggest-model.zip")

print("\n" + "="*50)
print("NEXT STEPS:")
print("="*50)
print(f"1. Unzip autosuggest-model.zip")
print(f"2. cd into the unzipped folder")
print(f"3. ollama create {ollama_model_name} -f Modelfile")
print(f"4. Open AutoSuggest > Settings > Model")
print(f"5. Set Ollama model to: {ollama_model_name}")
print(f"6. Enjoy your personalized autocomplete!")

---

## Optional: Download LoRA Adapter Only

If you want to keep the adapter separately (smaller download, can merge later):

In [ ]:
# Optional: download just the LoRA adapter
shutil.make_archive("autosuggest-adapter", "zip", "output/adapter")
files.download("autosuggest-adapter.zip")
print("Adapter downloaded! You can merge it locally later.")